# 본선 — LightGBM 학습 및 ID별 집계 제출

`01_signal_eda_and_feature_engineering.ipynb` 에서 저장한 슬라이딩 윈도우 데이터셋으로 LightGBM 회귀를 학습합니다.

테스트 시점에는 한 파일(ID) 당 여러 윈도우 행이 생기므로, 윈도우별 예측을 ID 단위로 **24.5 percentile** 로 집계해 제출합니다. 라벨이 큰 단위(round to nearest 100) 로 주어지므로 마지막에 `round(-2)` 후처리합니다.


## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

warnings.filterwarnings("ignore")
SEED = 42

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed" / "finals"
SUBMISSION_DIR = Path("submissions")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

## 처리된 데이터 로드

In [ ]:
X_train = pd.read_parquet(PROCESSED_DIR / "X_train.parquet")
X_valid = pd.read_parquet(PROCESSED_DIR / "X_valid.parquet")
X_test = pd.read_parquet(PROCESSED_DIR / "X_test.parquet")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

y_train = X_train.pop("label")
y_valid = X_valid.pop("label")
test_ids = X_test.pop("ID")
print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}, X_test: {X_test.shape}")

## 모델 학습

In [ ]:
model = LGBMRegressor(
    n_estimators=10000,
    learning_rate=0.05,
    num_leaves=65,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    verbose=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="mae",
    callbacks=[early_stopping(stopping_rounds=200), log_evaluation(500)],
)
val_pred = np.clip(model.predict(X_valid), 0, None)
val_mae = mean_absolute_error(y_valid, val_pred)
print(f"validation MAE (window-level): {val_mae:.4f}")

## 피처 중요도

In [ ]:
importance = (
    pd.DataFrame({"feature": X_train.columns, "importance": model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importance["feature"][::-1], importance["importance"][::-1], color="steelblue")
ax.set_title("LightGBM feature importance")
plt.tight_layout()
plt.show()

## 테스트 예측 + ID 단위 집계

윈도우 단위 예측을 ID 별로 24.5 percentile 로 집계합니다. percentile 값은 학습 셋의 검증 집계 결과를 기준으로 선택했습니다(작은 percentile 일수록 안정적인 하한 추정에 가깝습니다).


In [ ]:
test_pred = np.clip(model.predict(X_test), 0, None)
test_frame = pd.DataFrame({"ID": test_ids, "weight": test_pred})

aggregated = (
    test_frame.groupby("ID")["weight"]
    .agg(lambda values: np.percentile(values, 24.5))
    .reset_index()
)

submission = sample_submission.merge(aggregated, on="ID", how="left", suffixes=("", "_pred"))
submission["weight"] = submission["weight_pred"].fillna(submission["weight"]) if "weight" in submission.columns else submission["weight_pred"]
submission = submission[["ID", "weight"]].copy()
submission["weight"] = submission["weight"].apply(lambda value: round(value, -2))

output_path = SUBMISSION_DIR / "finals_lgbm_submission.csv"
submission.to_csv(output_path, index=False)
print(f"saved: {output_path}")
submission.head()

## 정리

- 본선은 시계열 신호 → trial 단위 weight 회귀 문제로, **윈도우 단위 학습 → ID 단위 집계** 흐름을 사용했습니다.
- 학습은 LightGBM + early stopping(200) 으로 단일 모델만 사용했습니다 (예선의 dual ensemble 보다 단순한 구조).
- 윈도우별 예측을 24.5 percentile 로 집계하고, 라벨 단위에 맞춘 round(-2) 후처리를 거쳐 제출 파일을 생성했습니다.
- 추가 개선 여지: window 길이 / percentile 임계값 / multi-seed 평균 / 1D-CNN 또는 Transformer 기반 시계열 모델 비교.
